# Course Cancellation Prediction

Collaborative academic Machine Learning project by Gal and Aviv.

Original datasets are not included. To run this notebook, provide the training and test CSV files in the working directory.

**חלק א'**

בתא זה אנו טוענים את כל הספריות הדרושות לביצוע הפרוייקט על פי CRISP-DM, הכוללת ספריות לניהול נתונים, ויזואליזציה, הכנת נתונים ואימון המודלים.

In [ ]:
#LIBRARIES LOADING
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from sklearn.metrics import RocCurveDisplay
import sys
import warnings
if not sys.warnoptions:
    warnings.simplefilter("ignore")

In [ ]:
train = pd.read_csv('Train_Data.csv')
test = pd.read_csv('Test_Data_No_Target.csv')
pd.set_option("display.max_columns", None)
train.head()

In [ ]:
test.shape

In [ ]:
train.shape

סט האימון- סט הנתונים כולל 63,464 תצפיות ו-29 משתנים.
סט הדיקה- כולל 15,866 תצפיות ו-28 משתנים.  משתנה המטרה אינו מופיע בסט זה, ומבנה העמודות האחרות תואם לחלטין לסט האימון.


In [ ]:
train.info()

הפונקציה מציגה את סוג בנתונים של כל עמודה ואת כמות הערכים שאינם ריקים הקיימים בה, בנוסף היא מספקת לנו תובנות חשובות למבנה הנתונים.
רוב המשתנים במערך הנתונים הם מספריים, משתנים אלו כוללים נתונים כגון כמויות, ימי המתנה, רישום, ממדי עלות וזמן ועוד.
במערך נתונים 8 משנתים קטגוריאליים , משנים אלו מייצגים מאפיינם איכותיים כגון מדינת המקור, חבילות אירוח והמתנות, מאפיינים טכנים ותפעוליים, תנאי תשלום ותאריך פתיחת קורס.
משתנים אלו נרצה להפוך למשתנים מספריים על מנת שנוכל לנתח אותם במודל.
משתנה המטרה הוא כבר משתנה מספרי בינארי המחזיר ערכים של 0 או 1.

In [ ]:
#נבדוק אילו ערכים חסרים
train.isna().sum().to_frame(name="number_of_missing_values").sort_values("number_of_missing_values", ascending=False)

 מתוך המשתנים המכילים ערכים חסרים, קיימים משתנים מספריים (כמו ימי רישום מראש, עלות לימוד יומית ומספר סטודנטים) ומשתנים קטגוריאליים (כמו מדינת מקור, סוג רישום ותנאי תשלום). בשלבי ניקוי הנתונים נטפל במשתנים המספריים באמצעות השלמה לפי החציון (Median), ובמשתנים הקטגוריאליים באמצעות הגדרת קטגוריה חדשה בשם 'Unknown' כדי לשמור על המידע ולא לייצר הטיה.

In [ ]:
train.describe()

פונקציית ה-describe מאפשרת לנו לבחון את המדדים הסטטיסטיים המרכזיים של כל המשתנים המספריים ומספקת תובנות קריטיות לגבי התפלגות הנתונים:

*   סקאלת המשתנים - ניתן לראות שקיים שוני משמעותי בין המשתנים השונים. ישנם משתנים שנעים בטווחים קטנים מאוד וישנם משתנים שנעים בטווחים שמגיעים לאלפים. הבדל זה מצביע על כך שבשלב הכנת הנתונים נצטרך אולי לבצע נרמול למשתנים(תלוי בסוג המודל) , כדי למנוע   מהמודל לתת משקל יתר למשתנים בעלי ערכים גדולים באופן מלאכותי.
*   זיהוי ערכים חריגים וחשד לשגיאות - מבט על ערכי המקסימום והמינימום חושף מספר אנומליות קיצוניות בדאטה שיידרשו טיפול וניקוי בשלבים הבאים: 1.  במשתנה של שעות מעשיות קיים ערך מינימום שלילי (-5) וערך מקסימום קיצוני של 10000 , בעוד שחציון השעות עומד על 1 בלבד.
2. במשתנה של ספירת הסטודנטים קיים ערך מקסימלי קיצוני של 9,999.
*   איזון משתנה המטרה - משורת הממוצע אנו רואים שערכו עומד על 0.414, מה שמעיד על כך שהדאטה מאוזנת יחסית .






In [ ]:
plt.rcParams.update({"axes.titlesize": 9})
train.hist(figsize=(20,20), xlabelsize=5, ylabelsize=5)
plt.show()

פקנו היסטוגרמות עבור כל המשתנים המספריים במערך הנתונים במטרה לבחון ויזואלית את אופן התפלגות הערכים שלהם.
**תובנות מרכזיות מתוך הגרפים:**
1. זוהו משתנים המכילים ערכים קיצוניים אשר ייבחנו בהמשך במסגרת ניתוח חריגים:ויזואליות של הגרפים של המשתנים כמו שעות מעשיות וספירת סטודנטים מאששים את מה שראינו בפונקציית discribe על כך שקיימים ערכים קיצוניים וחריגים מאוד שמותחים את הסקאלה של הדרף.
2. משתנים דלילים- ניתן לראות כי חלק מהמשתים המספריים מרוכזים כמעט לחלוטין סביב הערך 0 עם שונות נמוכה מאוד, דבר המרמז על כך שהם עשויים להיות דלילים ולא לתרום מידע קריטי למודל.


In [ ]:
#נבדוק את משתנה המטרה
plt.figure(figsize=(6, 4))
sns.countplot(data=train, x='Dropped_Course', palette='Set2')
plt.title('Distribution of the Target Variable (Dropped Course)', fontsize=12, fontweight='bold')
plt.xlabel('Dropped Course (0 = Kept Course, 1 = Dropped Course)', fontsize=10)
plt.ylabel('Count of Observations', fontsize=10)
plt.show()

גרף העמודות המוצג לעיל מבודד את משתנה המטרה ומאפשר לנו לבחון בצורה מדויקת את התפלגות המחלקות בפרויקט.

ניתוח ומסקנות מתוך הויזואליזציה:
1.
תמונת מצב כמותית: הגרף מציג באופן ברור שתי עמודות (עבור 0 ו-1) שגובהן קרוב יחסית. מתוך 63,464 התצפיות בסט האימון, הקבוצה ששמרה על ההרשמה (ערך 0) עומדת על קרוב ל-37,000 תצפיות, בעוד שהקבוצה שביטלה את הקורס (ערך 1) מייצגת כ-26,000 תצפיות.
2.
מוכנות לאימון המודל : כפי שראינו, היחס עומד על כ-41.4% מול כ-58.6%. בעולם של למידת מכונה, חלוקה זו מציגה בצורה גרפית וברורה את היותו של בסיס הנתונים מאוזן ומוכן לאימון . המשמעות היא שאין לנו הטיה קיצונית לטובת אחת המחלקות, ולכן המודלים שנאמן יוכלו ללמוד את שתי התוצאות בצורה שווה ויעילה, ללא צורך בהפעלת מניפולציות מלאכותיות על הדאטה.

In [ ]:
#בחינת השפעת נאמנות הלקוח (Returning_Client) על ההסתברות לביטול.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
avg_returning = train.groupby('Dropped_Course')['Returning_Client'].mean().reset_index()
sns.barplot(data=avg_returning, x='Dropped_Course', y='Returning_Client', ax=axes[0], palette='Set2')
axes[0].set_title('Average Returning Client Rate', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Current Dropped Course (0=No, 1=Yes)')
axes[0].set_ylabel('Mean of Returning_Client')
sns.countplot(data=train, x='Returning_Client', hue='Dropped_Course', ax=axes[1], palette='Set2')
axes[1].set_title('Returning Client vs Dropped Course', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Returning_Client (0=New, 1=Returning)')
axes[1].set_ylabel('Count of Observations')
plt.tight_layout()
plt.show()

השפעת משתנה הנאמנות (Returning_Client): לקוחות חדשים (ערך 0) נוטים לבטל הזמנות בשיעור גבוה משמעותית בהשוואה ללקוחות חוזרים (ערך 1). מבחינה עסקית, לקוחות שכבר צברו ניסיון חיובי מול האקדמיה מפגינים נאמנות גבוהה ומייצגים סיכון נמוך, בעוד שכלקוח חדש נכנס למערכת קיים "סיכון הסתגלות" גבוה המצריך מעקב תפעולי הדוק יותר.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
avg_reg = train.groupby('Dropped_Course')['Registration_Days_Before'].mean().reset_index()
sns.barplot(data=avg_reg, x='Dropped_Course', y='Registration_Days_Before', ax=axes[0], palette='Set2')
axes[0].set_title('Average Registration Days Before', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Dropped Course (0=No, 1=Yes)')
axes[0].set_ylabel('Mean Days Before')
sns.boxplot(data=train, x='Dropped_Course', y='Registration_Days_Before', ax=axes[1], palette='Set2')
axes[1].set_title('Distribution & Outliers of Registration Days', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Dropped Course (0=No, 1=Yes)')
axes[1].set_ylabel('Registration Days Before')
plt.tight_layout()
plt.show()

קשר בין מועד ההרשמה לבין ביטול הקורס- נמצא הבדל משמעותי במשתנה Registration_Days_Before, המייצג את מספר הימים שעברו ממועד ההרשמה ועד פתיחת הקורס.
כדי לקבל תמונה מלאה, הפקנו שני סוגי גרפים - Boxplotu ו-Barplot:
*   הבדל משמעותי בממוצעים: נמצא הבדל משמעותי בין שתי הקבוצות. קבוצות שלא ביטלו את השתתפותן (ערך 0) נרשמו בממוצע כ-70.6 ימים לפני פתיחת הקורס, בעוד שקבוצות שביטלו את ההרשמה (ערך 1) נרשמו בממוצע כ-148.5 ימים מראש - פער של יותר מפי 2 בזמן.
* פיזור הנתונים : גרף הקופסה מראה כי ערכי Registration_Days_Before של קבוצות שביטלו את הקורס מרוכזים ברמות גבוהות יותר בהשוואה לקבוצות שלא ביטלו. מיקום הקופסה והחציון מצביעים על כך שהתופעה אינה נובעת ממספר קטן של מקרים חריגים בלבד, אלא מאפיינת חלק משמעותי מהתצפיות.
תובנה:
הממצא מצביע על קשר אפשרי כי ככל שההרשמה מתבצעת זמן רב יותר לפני תחילת הקורס, כך גדל הסיכוי לביטולו.
הסבר הגיוני לכך הוא שככל שמועד הקורס רחוק יותר בלוח השנה, רמת אי-הוודאות בתוך הארגון הלקוח היא גבוהה יותר. לאורך החודשים הללו עלולים להתרחש שינויים תפעוליים, תזוזות בתקציבים עתידיים או שינויים פנימיים בצרכי כוח האדם של החברה, מה שמוביל בסופו של דבר להחלטה על ביטול ההרשמה לקורס.


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
avg_dropouts = train.groupby('Dropped_Course')['Prev_Course_Dropouts'].mean().reset_index()
sns.barplot(data=avg_dropouts, x='Dropped_Course', y='Prev_Course_Dropouts', ax=axes[0], palette='Set2')
axes[0].set_title('Average Previous Dropouts', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Current Dropped Course (0=No, 1=Yes)')
axes[0].set_ylabel('Mean of Previous Dropouts')
sns.boxplot(data=train, x='Dropped_Course', y='Prev_Course_Dropouts', ax=axes[1], palette='Set2')
axes[1].set_title('Distribution of Previous Dropouts', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Current Dropped Course (0=No, 1=Yes)')
axes[1].set_ylabel('Previous Dropouts')
plt.tight_layout()
plt.show()

השפעת היסטוריית ביטולים קודמת- נמצא קשר בין מספר הביטולים הקודמים של הקבוצה (Prev_Course_Dropouts) לבין ההסתברות לביטול הקורס הנוכחי. קבוצות שלא ביטלו את הקורס הנוכחי ביצעו בממוצע כ-0.02 ביטולים בעבר, בעוד שקבוצות שביטלו את הקורס הנוכחי ביצעו בממוצע כ-0.20 ביטולים בעבר.
ממצא זה מצביע על כך שהתנהגות עבר מהווה אינדיקטור משמעותי להתנהגות עתידית. לקוחות בעלי היסטוריה של ביטולים נוטים יותר לבטל גם הרשמות עתידיות.
תובנה:
ממצא זה מצביע על כך שהתנהגות עבר מהווה אינדיקטור משמעותי להתנהגות עתידית. לקוחות בעלי היסטוריה של ביטולים נוטים יותר לבטל גם הרשמות עתידיות.

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

avg_attended = train.groupby('Dropped_Course')['Prev_Course_Attended'].mean().reset_index()
sns.barplot(data=avg_attended, x='Dropped_Course', y='Prev_Course_Attended', ax=axes[0], palette='Set2')
axes[0].set_title('Average Previous Courses Attended', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Current Dropped Course (0=No, 1=Yes)')
axes[0].set_ylabel('Mean of Previous Courses Attended')

sns.boxplot(data=train, x='Dropped_Course', y='Prev_Course_Attended', ax=axes[1], palette='Set2')
axes[1].set_title('Distribution of Previous Courses Attended', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Current Dropped Course (0=No, 1=Yes)')
axes[1].set_ylabel('Previous Courses Attended')

plt.tight_layout()
plt.show()


השפעת השתתפות בקורסים קודמים- נמצא קשר בין מספר הקורסים הקודמים שבהם השתתפה הקבוצה (Prev_Course_Attended) לבין ההסתברות לביטול.
קבוצות שלא ביטלו את הקורס הנוכחי השתתפו בממוצע בכ-0.19 קורסים קודמים, בעוד שקבוצות שביטלו השתתפו בממוצע בכ-0.03 קורסים בלבד. ממצא זה עשוי להעיד על נאמנות לקוחות ועל שביעות רצון גבוהה יותר בקרב לקוחות חוזרים. לקוחות שכבר השתתפו בקורסים קודמים מכירים את החברה ואת שירותיה ולכן נוטים פחות לבטל הרשמות עתידיות.
תובנה:
ממצא זה עשוי להעיד על נאמנות לקוחות ועל שביעות רצון גבוהה יותר בקרב לקוחות חוזרים. לקוחות שכבר השתתפו בקורסים קודמים מכירים את החברה ואת שירותיה, ולכן נוטים פחות לבטל הרשמות עתידיות.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
avg_waiting = train.groupby('Dropped_Course')['Waiting_List_Days'].mean().reset_index()
sns.barplot(data=avg_waiting, x='Dropped_Course', y='Waiting_List_Days', ax=axes[0], palette='Set2')
axes[0].set_title('Average Waiting List Days', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Current Dropped Course (0=No, 1=Yes)')
axes[0].set_ylabel('Mean of Waiting List Days')
sns.boxplot(data=train, x='Dropped_Course', y='Waiting_List_Days', ax=axes[1], palette='Set2')
axes[1].set_title('Distribution of Waiting List Days', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Current Dropped Course (0=No, 1=Yes)')
axes[1].set_ylabel('Waiting List Days')

plt.tight_layout()
plt.show()

השפעת זמן ההמתנה ברשימת המתנה- נמצא הבדל במשתנה Waiting_List_Days, המייצג את מספר הימים שהקבוצה שהתה ברשימת המתנה. קבוצות שלא ביטלו המתינו בממוצע כ-2.66 ימים, בעוד שקבוצות שביטלו המתינו בממוצע כ-5.85 ימים. ממצא זה עשוי להעיד כי זמן המתנה ארוך פוגע בחוויית הלקוח ומגדיל את הסיכוי לביטול ההרשמה. ייתכן שבמהלך תקופת ההמתנה הלקוחות מוצאים פתרונות חלופיים או מאבדים עניין בקורס.
תובנה:
ממצא זה עשוי להעיד כי זמן המתנה ארוך פוגע בחוויית הלקוח ומגדיל את הסיכוי לביטול ההרשמה. ייתכן שבמהלך תקופת ההמתנה הלקוחות מוצאים פתרונות חלופיים אצל חברות מתחרות, או פשוט מאבדים עניין בקורס עקב חוסר ודאות

נעשה סדר בסוגי משתנים החסרים כדי להחליט באיזה דרך הכי נכון להשלים אותם:


In [ ]:
# בדיקת ערכים חסרים בכל עמודה
missing_table = pd.DataFrame({"Missing_Count": train.isna().sum(),"Missing_Percent": train.isna().mean() * 100})

missing_table = missing_table[missing_table["Missing_Count"] > 0]
missing_table = missing_table.sort_values(by="Missing_Percent", ascending=False)

missing_table

In [ ]:
plt.figure(figsize=(12, 8))
# נבחר רק את העמודות המספריות הרלוונטיות (נשמיט מזהים כמו Client_ID)
numeric_cols = train.select_dtypes(include=['int64', 'float64']).drop(columns=['Client_ID', 'Agent_ID', 'Company_ID'], errors='ignore')

corr_matrix = numeric_cols.corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix of Numeric Features')
plt.show()

In [ ]:
#בדיקת סוגי המשתנים שבהם קיימים ערכים חסרים
train[missing_table.index].dtypes


In [ ]:
# בדיקת הערכים הנפוצים בעמודות קטגוריות עם חוסרים

missing_cat_cols = ['Origin_Country','Catering_Package','Requested_Lab_Config','Enrollment_Type','Submission_Source','Payment_Terms']

for col in missing_cat_cols:
    print("\n" + "="*50)
    print(col)
    print("Missing values:", train[col].isna().sum())
    print("Missing percent:", round(train[col].isna().mean() * 100, 2), "%")
    print("Unique values:", train[col].nunique())
    print(train[col].value_counts(dropna=False).head(10))

בשלב זה בדקנו אילו עמודות מכילות ערכים חסרים ומה שיעור החוסרים בכל עמודה. מטרת הבדיקה היא להבין האם מדובר בכמות קטנה של חוסרים שניתן להשלים בצורה פשוטה, או בעמודות בעייתיות שבהן שיעור החוסרים גבוה ודורש החלטה מיוחדת.

מהטבלה ניתן לראות כי רוב העמודות מכילות שיעור חוסרים נמוך יחסית, אך קיימות שתי עמודות חריגות: Company_IDעם כ־95% ערכים חסרים ו- Agent_ID עם כ-17.6% ערכים חסרים. לכן, עבור Company_ID לא נכון לבצע השלמה רגילה, משום שרוב המידע בעמודה חסר. לגבי Agent_ID יש לבחון בהמשך האם החוסר עצמו עשוי לשאת מידע או שיש להתייחס אליו כקטגוריה נפרדת.

בנוסף, בבדיקת העמודות הקטגוריות נמצא כי קיימות קטגוריות שמופיעות בכמה צורות שונות, למשל בשל אותיות גדולות/קטנות ורווחים מיותרים. לכן, לפני השלמת הערכים החסרים בעמודות הקטגוריות, יש לבצע ניקוי טקסט ואיחוד קטגוריות זהות.

In [ ]:
# ניקוי
categorical_cols_to_clean = ['Origin_Country','Catering_Package','Welcome_Gift_Type','Requested_Lab_Config','Assigned_Lab_Config','Enrollment_Type','Lanyard_Color','Client_Category','Submission_Source','Payment_Terms']

for col in categorical_cols_to_clean:
    train[col] = train[col].str.strip().str.lower()
    test[col] = test[col].str.strip().str.lower()

בבדיקת הערכים הקטגוריים נמצא כי חלק מהקטגוריות מופיעות בכמה וריאציות שונות, למשל בשל רווחים בתחילת/סוף הטקסט או שימוש שונה באותיות גדולות וקטנות. מצב זה עלול לגרום למודל להתייחס לאותה קטגוריה כאל מספר קטגוריות שונות, ולכן לפני השלמת ערכים חסרים ואימון המודל ביצענו סטנדרטיזציה של הערכים הקטגוריים: הסרת רווחים מיותרים והמרה לאותיות קטנות.

In [ ]:
#בדיקה לאחר ניקוי -train
for col in missing_cat_cols:
    print("\n" + "="*50)
    print(col)
    print("Unique values after cleaning:", train[col].nunique())
    print(train[col].value_counts(dropna=False).head(10))

In [ ]:
#בדיקה לאחר ניקוי -test
for col in missing_cat_cols:
    print("\n" + "="*50)
    print(f"{col} (TEST)")
    print("Unique values after cleaning:", test[col].nunique())
    print(test[col].value_counts(dropna=False).head(10))

לאחר ביצוע סטנדרטיזציה של אותיות ורווחים, מספר הקטגוריות הייחודיות ירד באופן משמעותי במספר משתנים. יחד עם זאת, נמצאו עדיין קטגוריות חריגות המכילות תווים מיוחדים כגון #, *, ! ו-?, העשויים להעיד על בעיות איכות נתונים נוספות. ממצאים אלו יתועדו וייבחנו במסגרת שלב Feature Engineering וניתוח איכות הנתונים.

In [ ]:
# טבלת מילוי חוסרים

missing_decisions_data = [
    {
        "Column": "Company_ID",
        "Missing_Percent": 95.08,
        "Variable_Type": "Identifier",
        "Decision": "Exclude from modeling",
        "Reason": "Very high missing rate; regular imputation would not be reliable."
    },
    {
        "Column": "Agent_ID",
        "Missing_Percent": 17.61,
        "Variable_Type": "Identifier / Categorical",
        "Decision": "Fill missing values with 'Unknown'",
        "Reason": "Although numeric in format, this variable represents an ID rather than a continuous numeric value."
    },
    {
        "Column": "Registration_Days_Before",
        "Missing_Percent": 4.20,
        "Variable_Type": "Numeric",
        "Decision": "Fill with median",
        "Reason": "The variable is numeric and appears skewed, so the median is more robust than the mean."
    },
    {
        "Column": "Requested_Lab_Config",
        "Missing_Percent": 2.74,
        "Variable_Type": "Categorical",
        "Decision": "Fill missing values with 'Unknown'",
        "Reason": "Missing values may represent unavailable or unrecorded configuration information."
    },
    {
        "Column": "Physical_Course_Kits",
        "Missing_Percent": 1.64,
        "Variable_Type": "Numeric",
        "Decision": "Fill with median",
        "Reason": "The variable is numeric; median imputation is robust to unusual values."
    },
    {
        "Column": "Enrollment_Type",
        "Missing_Percent": 1.13,
        "Variable_Type": "Categorical",
        "Decision": "Fill missing values with 'Unknown'",
        "Reason": "Missing values are treated as a separate category to preserve possible information in the missingness."
    },
    {
        "Column": "Submission_Source",
        "Missing_Percent": 0.95,
        "Variable_Type": "Categorical",
        "Decision": "Fill missing values with 'Unknown'",
        "Reason": "Missing source information may still carry predictive value."
    },
    {
        "Column": "Payment_Terms",
        "Missing_Percent": 0.92,
        "Variable_Type": "Categorical",
        "Decision": "Fill missing values with 'Unknown'",
        "Reason": "Missing payment information is treated as a separate category."
    },
    {
        "Column": "Origin_Country",
        "Missing_Percent": 0.88,
        "Variable_Type": "Categorical",
        "Decision": "Fill missing values with 'Unknown'",
        "Reason": "Country is categorical, so missing values are assigned to an explicit unknown category."
    },
    {
        "Column": "Catering_Package",
        "Missing_Percent": 0.64,
        "Variable_Type": "Categorical",
        "Decision": "Fill missing values with 'Unknown'",
        "Reason": "Categorical missing values are preserved as a separate unknown category."
    },
    {
        "Column": "Daily_Tuition_Cost",
        "Missing_Percent": 0.12,
        "Variable_Type": "Numeric",
        "Decision": "Fill with median",
        "Reason": "The variable contains outliers, so the median is preferred over the mean."
    },
    {
        "Column": "Students_Count",
        "Missing_Percent": 0.006,
        "Variable_Type": "Numeric",
        "Decision": "Fill with median",
        "Reason": "Very small missing rate; median imputation has negligible impact."
    }
]

missing_decisions = pd.DataFrame(missing_decisions_data)

missing_decisions

טבלה זו מסכמת את ההחלטות שהתקבלו עבור עמודות המכילות ערכים חסרים. עבור משתנים מספריים נבחרה לרוב השלמה באמצעות חציון, מכיוון שחלק מהמשתנים מציגים התפלגות מוטה וערכים חריגים, ולכן החציון יציב יותר מהממוצע. עבור משתנים קטגוריים נבחרה השלמה באמצעות הערך "unknown", כדי לשמר את האפשרות שהחוסר עצמו מכיל מידע ולא למחוק תצפיות. עבור Company_ID, שבו שיעור החוסרים גבוה במיוחד, הוחלט שלא להשתמש בעמודה בשלב המידול, משום שהשלמה רגילה אינה אמינה כאשר רוב הערכים חסרים.

In [ ]:
#לצורך השוואת התפלגויות לפני ואחרי השלמת הערכים החסרים, נטענה מחדש גרסת הנתונים המקורית ללא השלמות.
train_original = pd.read_csv('Train_Data.csv')
test_original = pd.read_csv('Test_Data_No_Target.csv')

In [ ]:
# בחירת העמודות הנומריות בלבד לצורך חישוב קורלציה
numeric_cols = train.select_dtypes(include=['int64', 'float64']).columns

plt.figure(figsize=(12, 8))
correlation_matrix = train[numeric_cols].corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Correlation Matrix for Numeric Features and Target')
plt.show()

ממטריצת הקורלציה ניתן לראות את עוצמת הקשר
הליניארי בין המשתנים השונים. אנו מחפשים משתנים עם מתאם גבוה למשתנה המטרה Dropped_Course כדי להבין מי מהם עשוי להיות בעל כוח ניבוי חזק במודל.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x='Dropped_Course', y='Registration_Days_Before', data=train, palette='Set2')
plt.title('Registration Days Before Course by Drop Status')
plt.xlabel('Dropped Course (0 = No, 1 = Yes)')
plt.ylabel('Registration Days Before')
plt.show()


מגרף הקופסה המציג את Registration_Days_Before ביחס ל-Dropped_Course, ניתן לראות האם קיים שוני בהתפלגות ימי ההרשמה בין סטודנטים שהשלימו את הקורס לבין כאלו שנשרו

In [ ]:
#השלמת Students_Count
students_median = train['Students_Count'].median()
train['Students_Count'] = train['Students_Count'].fillna(students_median)
test['Students_Count'] = test['Students_Count'].fillna(students_median)

עמודת Students_Count מכילה שיעור חוסרים זניח (כ-0.006%).
מאחר שמדובר במשתנה מספרי המכיל ערכים חריגים, נבחרה השלמה באמצעות החציון (Median),
אשר פחות מושפע מערכים קיצוניים בהשוואה לממוצע.

In [ ]:
#השלמת Registration_Days_Before
registration_median = train['Registration_Days_Before'].median()
train['Registration_Days_Before'] = train['Registration_Days_Before'].fillna(registration_median)
test['Registration_Days_Before'] = test['Registration_Days_Before'].fillna(registration_median)

עמודת Registration_Days_Before מכילה כ-4.2% ערכים חסרים. מכיוון שההתפלגות אינה סימטרית וקיימים ערכים גבוהים יחסית, נבחרה השלמה באמצעות החציון, המספק אומדן יציב יותר מהממוצע.

In [ ]:

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
sns.histplot(train_original['Registration_Days_Before'].dropna(), bins=30)
plt.title("Before Imputation")

plt.subplot(1,2,2)
sns.histplot(train['Registration_Days_Before'], bins=30)
plt.title("After Imputation")

plt.tight_layout()
plt.show()

ניתן לראות כי לאחר השלמת הערכים החסרים באמצעות החציון, נוספה מסה קטנה של תצפיות סביב ערך החציון. עם זאת, צורת ההתפלגות הכללית נשמרה ולא נוצרה הטיה משמעותית בנתונים. לכן, השימוש בחציון מהווה שיטת השלמה מתאימה עבור משתנה זה.

In [ ]:
 #השלמת Physical_Course_Kits
kits_median = train['Physical_Course_Kits'].median()
train['Physical_Course_Kits'] = train['Physical_Course_Kits'].fillna(kits_median)
test['Physical_Course_Kits'] = test['Physical_Course_Kits'].fillna(kits_median)

עמודת Physical_Course_Kits היא משתנה מספרי בעל שיעור חוסרים נמוך יחסית. הערכים החסרים במשתנה זה הושלמו באמצעות מדד החציון (Median), על מנת לצמצם השפעה אפשרית של ערכים חריגים ולשמור על התפלגות הנתונים המקורית.

כדי למנוע זליגת נתונים (Data Leakage), ערך החציון חושב על סמך נתוני ה-Train בלבד, ויושם בצורה אחידה הן על קובץ ה-Train והן על קובץ ה-Test. אסטרטגיה זו מבטיחה כי המבנה של שני מאגרי הנתונים יישאר זהה לחלוטין לקראת שלב בניית המודלים

In [ ]:
#השלמת Daily_Tuition_Cost
tuition_median = train['Daily_Tuition_Cost'].median()
train['Daily_Tuition_Cost'] = train['Daily_Tuition_Cost'].fillna(tuition_median)
test['Daily_Tuition_Cost'] = test['Daily_Tuition_Cost'].fillna(tuition_median)

עמודת Daily_Tuition_Cost מכילה מספר קטן של ערכים חסרים.
 מאחר שנמצאו ערכים חריגים בהתפלגות המשתנה, נבחרה השלמה באמצעות החציון, אשר מייצג בצורה טובה יותר את הערך הטיפוסי.

In [ ]:
#השלמת Origin_Country
train['Origin_Country'] = train['Origin_Country'].fillna('Unknown')
test['Origin_Country'] = test['Origin_Country'].fillna('Unknown')

Origin_Country הוא משתנה קטגוריאלי.
הערכים החסרים הוחלפו בקטגוריה "Unknown", על מנת לשמור את המידע על עצם החוסר ולא לכפות שיוך למדינה קיימת.

In [ ]:
#השלמת Catering_Package
train['Catering_Package'] = train['Catering_Package'].fillna('Unknown')
test['Catering_Package'] = test['Catering_Package'].fillna('Unknown')

בעמודת Catering_Package הוחלט להחליף את הערכים החסרים בקטגוריה "Unknown".
גישה זו מאפשרת למודל ללמוד האם עצם החוסר במידע קשור להתנהגות הלקוחות.

In [ ]:
#השלמת Requested_Lab_Config
train['Requested_Lab_Config'] = train['Requested_Lab_Config'].fillna('Unknown')
test['Requested_Lab_Config'] = test['Requested_Lab_Config'].fillna('Unknown')

עמודת Requested_Lab_Config היא משתנה קטגוריאלי.
הערכים החסרים הוגדרו כ-"Unknown" כדי לשמר מידע אפשרי הגלום בעצם היעדר הנתון.

In [ ]:
#השלמת Enrollment_Type
train['Enrollment_Type'] = train['Enrollment_Type'].fillna('Unknown')
test['Enrollment_Type'] = test['Enrollment_Type'].fillna('Unknown')

עמודת Enrollment_Type מכילה מספר קטן של ערכים חסרים.
בחרנו להשלים אותם באמצעות קטגוריה נפרדת ("Unknown") במקום להשתמש בערך השכיח.

In [ ]:
#השלמת Submission_Source
train['Submission_Source'] = train['Submission_Source'].fillna('Unknown')
test['Submission_Source'] = test['Submission_Source'].fillna('Unknown')

הערכים החסרים בעמודת Submission_Source הוחלפו בקטגוריה "Unknown" על מנת לשמר מידע אפשרי הנובע מהיעדר מקור הרשמה מתועד.

In [ ]:
#השלמת Payment_Terms
train['Payment_Terms'] = train['Payment_Terms'].fillna('Unknown')
test['Payment_Terms'] = test['Payment_Terms'].fillna('Unknown')

עמודת Payment_Terms היא משתנה קטגורי ולכן הערכים החסרים הוחלפו בקטגוריה "Unknown". כך ניתן לשמר מידע על לקוחות עבורם תנאי התשלום אינם ידועים.

In [ ]:
#השלמת Agent_ID
train['Agent_ID'] = train['Agent_ID'].astype('object')
test['Agent_ID'] = test['Agent_ID'].astype('object')

train['Agent_ID'] = train['Agent_ID'].fillna('Unknown')
test['Agent_ID'] = test['Agent_ID'].fillna('Unknown')

למרות ש-Agent_ID מיוצג כמספר, מדובר למעשה במזהה ולא במשתנה מספרי רציף. לכן הוחלט להתייחס אליו כמשתנה קטגורי ולהחליף ערכים חסרים ב-"Unknown".

עמודת Company_ID מכילה כ-95% ערכים חסרים.
 בשל שיעור החוסרים הגבוה במיוחד, הוחלט שלא לבצע השלמה לעמודה זו ולבחון את הסרתה בשלב בניית המודלים.

In [ ]:
train.isna().sum().sort_values(ascending=False).head(20)

In [ ]:
test.isna().sum().sort_values(ascending=False).head(20)

לאחר ביצוע השלמת הערכים החסרים, בוצעה בדיקה נוספת של כלל המשתנים במאגר הנתונים. נמצא כי כל הערכים החסרים טופלו בהצלחה, למעט בעמודת Company_ID.

עמודת Company_ID מכילה כ־95% ערכים חסרים, ולכן הוחלט שלא לבצע לה השלמה. שיעור חוסרים כה גבוה עלול להוביל להשלמות לא אמינות ולפגיעה באיכות הנתונים. לפיכך, עמודה זו תיבחן בהמשך התהליך כמועמדת להסרה בשלב בניית המודלים.

מלבד Company_ID, לא נותרו ערכים חסרים במאגר הנתונים.

** Part B - Outlier Analysis and Feature Engineering**

בחלק זה נבחן ערכים חריגים וניצור משתנים חדשים על בסיס התובנות שעלו בשלב ה-EDA. מטרת השלב היא לשפר את איכות הנתונים ואת יכולת המודל לזהות דפוסים הקשורים לביטול הקורס.

בשלב זה נבוצע זיהוי ערכים חריגים (Outliers) עבור המשתנים המספריים באמצעות שיטת IQR (Interquartile Range).

שיטה זו מבוססת על הרבעונים של ההתפלגות ואינה רגישה לערכים קיצוניים כמו הממוצע וסטיית התקן, ולכן היא מתאימה במיוחד לנתונים שאינם מתפלגים נורמלית.

תחילה חושבו:

- Q1 – הרבעון הראשון (25% מהתצפיות נמצאות מתחתיו)
- Q3 – הרבעון השלישי (75% מהתצפיות נמצאות מתחתיו)

לאחר מכן חושב הטווח הבין־רבעוני:

$$IQR = Q3 - Q1$$

לאחר חישוב ה-IQR הוגדרו גבולות לזיהוי ערכים חריגים:

$$Lower Bound = Q1 - 1.5 × IQR$$

$$Upper Bound = Q3 + 1.5 × IQR$$

כל תצפית שערכה קטן מהגבול התחתון או גדול מהגבול העליון סווגה כערך חריג.

להלן טבלת ריכוז המציגה את הגבולות שחושבו, כמות הערכים החריגים ואחוזם מכלל התצפיות עבור כל אחד מהמשתנים המספריים במאגר הנתונים:


In [ ]:
#נזהה את הערכים החריגים
outlier_cols = [
    'Students_Count',
    'Practical_Hours',
    'Theory_Hours',
    'Registration_Days_Before',
    'Prev_Course_Dropouts',
    'Prev_Course_Attended',
    'Waiting_List_Days',
    'Registration_Changes',
    'Daily_Tuition_Cost'
]

outlier_summary = []

for col in outlier_cols:
    Q1 = train[col].quantile(0.25)
    Q3 = train[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers_count = ((train[col] < lower_bound) | (train[col] > upper_bound)).sum()
    outliers_percent = outliers_count / len(train) * 100

    outlier_summary.append({
        "Column": col,
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "Lower_Bound": lower_bound,
        "Upper_Bound": upper_bound,
        "Outliers_Count": outliers_count,
        "Outliers_Percent": outliers_percent
    })

outlier_summary_df = pd.DataFrame(outlier_summary)
outlier_summary_df.sort_values(by="Outliers_Percent", ascending=False)

מתוצאות הבדיקה עולה כי במספר משתנים, כגון Registration_Changes, Prev_Course_Dropouts, Students_Count, Waiting_List_Days ו-Prev_Course_Attended, התקבלו ערכי Q1 ו-Q3 השווים ל-0, ולכן גם IQR שווה ל-0. כתוצאה מכך, כל ערך השונה מ-0 סומן כערך חריג לפי הגדרת השיטה. לאחר בחינת התפלגות המשתנים נמצא כי מצב זה נובע מריבוי ערכי 0 בנתונים ואינו מעיד בהכרח על שגיאות או תצפיות חריגות מבחינה עסקית. הפעלת קטימה אוטומטית במקרה זה תאפס בכוח את כל הערכים החיוביים ותמחוק לחלוטין מידע קריטי לגבי קבוצות המיעוט. ערכים הגבוהים מ-0 במשתנים אלו מייצגים התנהגות עסקית אמיתית ובעלת כוח ניבוי רב עבור המודל, ואינם מהווים שגיאות הזנה, ולכן נשמרו ללא שינוי


בנוסף, זוהו ערכים קיצוניים במספר משתנים, ובפרט ב-Practical_Hours, Theory_Hours, Daily_Tuition_Cost ו-Registration_Days_Before. ערכים אלו נמצאו רחוקים משמעותית מרוב התצפיות ולכן סומנו כחריגים על ידי השיטה. בשלב זה הערכים נשמרו במאגר הנתונים, מאחר שלא נמצאו ראיות חד-משמעיות לכך שמדובר בשגיאות מדידה או הזנה, והם עשויים לייצג מקרים אמיתיים במערכת.

In [ ]:
train_raw = pd.read_csv('Train_Data.csv')

In [ ]:
#טיפול בערך חריג-Registration_Days_Before
train['Registration_Days_Before'] = np.clip(train['Registration_Days_Before'], -162.0, 326.0)
test['Registration_Days_Before'] = np.clip(test['Registration_Days_Before'], -162.0, 326.0)

טיפול בערך חריג-Registration_Days_Before
* החלטת טיפול: חסימה וקטימה לערך החסם העליון והתחתון של שיטת ה-IQR.
* המשתנה כולל קבוצה של סטודנטים שנרשמו תקופה ארוכה מאוד מראש (מעל 326 ימים). מכיוון שמדובר בכמות לא מבוטלת של תצפיות (כ-4.7% מהנתונים), הסרתן מהמדגם תוביל לאובדן מידע יקר ערך ותפגע ביכולת הלמידה וההכללה של המודל. מנגד, השארת ערכים קיצוניים אלו כפי שהם עלולה לעוות את סקאלת המשתנה ולהשפיע לרעה על אלגוריתמים הרגישים למרחקים או מבוססי רגרסיה. לכן, בחרנו להשתמש בשיטת הקטימה  ולהגביל את הערכים החריגים לגבולות החסמים.

In [ ]:
#טיפול בערך חריג-Daily_Tuition_Cost
train['Daily_Tuition_Cost'] = np.clip(train['Daily_Tuition_Cost'], 12.0, 180.0)
test['Daily_Tuition_Cost'] = np.clip(test['Daily_Tuition_Cost'], 12.0, 180.0)

#טיפול בערך חריג-Daily_Tuition_Cost
* החלטת טיפול: חסימה וקטימה לפי חסמי שיטת ה-IQR.
*  שכר הלימוד היומי כולל ערכים חריגים גם בקצה הנמוך  וגם בקצה הגבוה , המייצגים כ-4.5% מכלל התצפיות. על מנת להגן על המודלים מפני הטיה הנגרמת מערכים קיצוניים בשני הקצוות, ומבלי לצמצם את גודל המדגם על ידי מחיקת שורות, ביצענו חסימה של הערכים לטווח ה-IQR.

In [ ]:
#טיפול בערך חריג- Practical_Hours
train['Practical_Hours'] = np.clip(train['Practical_Hours'], -1.5, 2.5)
test['Practical_Hours'] = np.clip(test['Practical_Hours'], -1.5, 2.5)

#טיפול בערך חריג- Practical_Hours
* החלטת טיפול: חסימה וקטימה בטווח שבין 1.5- ל-2.5 שעות בהתאם לתוצאות שיטת ה-IQR.

*  משתנה זה הכיל ערכים קיצוניים , שהם בבירור שגיאות הזנה או הקלדה במערכת אשר עלולות להטות לחלוטין את פונקציות ההפסד של המודלים. בחרנו בקטימה  לערכי החסמים המחושבים על מנת לנטרל לחלוטין את השפעת השגיאות הקיצוניות הללו, תוך שמירה על שלמות התצפיות במדגם ומניעת אובדן נתונים.

#טיפול בערך חריג- Theory_Hours
למרות ששיטת ה-IQR סימנה ערכים מעל 6.0 שעות כחריגים (כ-0.89% מהנתונים), מבחינה עסקית ולוגית ערכים אלו הגיוניים לחלוטין ומייצגים קורסים מוגברים או ימי עיון מרוכזים. בניגוד למשתנה השעות המעשיות (שבו זוהו שגיאות מובהקות כמו 10,000 שעות), שעות התיאוריה מציגות ערכים ריאליים ואין אינדיקציה לכך שמדובר בשגיאות הזנה. לכן, נרצה שהמודל ילמד את המקרים האמיתיים הללו.
לכן הוחלט שלא לבצע קטימה למשתנה זה ולהשאיר את הערכים כפי שהם.

In [ ]:
#השוואה של הערכים החריגים לפני ואחרי הטיפול

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(y=train_raw['Practical_Hours'], ax=axes[0], color='lightcoral')
axes[0].set_title('Practical Hours - Before Capping (With 10,000h Error)')
axes[0].set_ylabel('Hours')
sns.boxplot(y=train['Practical_Hours'], ax=axes[1], color='lightgreen')
axes[1].set_title('Practical Hours - After Capping (IQR Bound 2.5)')
axes[1].set_ylabel('Hours')

plt.tight_layout()
plt.show()

* בגרף השמאלי (לפני הטיפול):ניתן לראות כיצד נוכחותו של ערך קיצוני בודד (10,000 שעות) מעוותת לחלוטין את סקאלת התצוגה של המשתנה ו"מועכת" את כל שאר הנתונים האמיתיים לתחתית הציר. ערך זה, לצד ערכים שליליים שנמצאו בדאטה, מהווה עדות חד-משמעית לשגיאות הזנה או הקלדה במערכת.

* בגרף הימני (לאחר הטיפול): ניתן לראות כיצד הפעלת פונקציית הקטימה לפי חסמי ה-IQR החזירה את המשתנה לטווח הלוגי והתקין שלו (בין 1.5- ל-2.5 שעות). כעת, מרכז ההתפלגות, החציון והטווח הבין-רבעוני מוצגים בצורה יציבה, נקייה ומאוזנת שאינה מוטה משגיאות הקיצון.

In [ ]:
# הוספת משתנה- Total_Participants
for df in [train, test]:
    df['Total_Participants'] = df['Students_Count'] + df['Professionals_Count'] + df['Observers_Count']

Total_Participants - משתנה זה סוכם את כמות הסטודנטים, אנשי המקצוע  והמשקיפים הרשומים בהזמנה כדי לייצג את גודל הקבוצה המלא, נעשה זאת משום שגודל הקבוצה הכולל משקף את מורכבות ההזמנה ועשוי להוות גורם מפתח בניבוי הסיכוי לביטולה או לנשירה מהקורס.

In [ ]:
# הוספת משתנה- Total_Course_Hours
for df in [train, test]:
    df['Total_Course_Hours'] = df['Theory_Hours'] + df['Practical_Hours']

 Total_Course_Hours - משתנה זה מחבר ומסכם את שעות התיאוריה (Theory_Hours) והשעות המעשיות (Practical_Hours) המציג את היקף השעות הכולל של הקורס. אורך הקורס ועומס הלימודים הכללי מהווים גורם מפתח המשפיע ישירות על כושר ההתמדה של הקבוצה ועל סיכויי הנשירה.

In [ ]:
#הוספת משתנה - Lab_Config_Match
for df in [train, test]:
    df['Lab_Config_Match'] = (df['Requested_Lab_Config'] == df['Assigned_Lab_Config']).astype(int)

Lab_Config_Match - משתנה בינארי המשווה בין סביבת המעבדה המבוקשת (Requested_Lab_Config) לזו שהוקצתה בפועל (Assigned_Lab_Config). המשתנה מקבל ערך 1 כאשר ישנה התאמה מלאה בין דרישת הקבוצה להקצאה בפועל, ו-0 במקרה של חוסר התאמה. נניח כי אי-התאמה טכנולוגית עלולה לייצר חוסר שביעות רצון אצל הלקוח ולשמש כגורם מזרז לנשירה מהקורס.

In [ ]:
#הוספת משתנה - Cost_Per_Participant
for df in [train, test]:
    df['Cost_Per_Participant'] = df['Daily_Tuition_Cost'] / (df['Total_Participants'] + 1)

Cost_Per_Participant - המשתנה הזה מחשב כמה כסף הושקע ביום על כל משתתף בודד בקבוצה (מחיר יומי כולל חלקי מספר האנשים). עשינו את זה כי הגיוני שככל שעלות ההשקעה לכל אדם גבוהה יותר, כך המחויבות לקורס תעלה והסיכוי לנשירה או ביטול ירד.

In [ ]:
#הוספת משתנה -Course_Start_Month
for df in [train, test]:
    df['Course_Start_Date'] = pd.to_datetime(df['Course_Start_Date'])
    df['Course_Start_Month'] = df['Course_Start_Date'].dt.month

Course_Start_Month- המשתנה הזה מחלץ את החודש מתוך תאריך פתיחת הקורס. עשינו את זה כדי שהמודל יוכל לזהות "עונתיות"– כלומר, לבדוק האם יש חודשים מסוימים בשנה (כמו חודשי הקיץ או החגים) שבהם קבוצות נוטות לבטל או לנשור יותר בהשוואה לחודשים אחרי

In [ ]:
#הוספת משתנה- Practical_Hours_Ratio
for df in [train, test]:
    df['Practical_Hours_Ratio'] = df['Practical_Hours'] / (df['Total_Course_Hours'] + 1)

המשתנה הזה מחשב את האחוז של שעות התרגול המעשי מתוך סך כל שעות הקורס. עשינו את זה כי אופי הקורס משפיע מאוד על שביעות הרצון ועל רמת העניין של המשתתפים, ולכן מהווה גורם מפתח בניבוי הסיכוי לנשירה או ביטול.

In [ ]:
#בידוד משתנה המטרה והפרדת הנתונים
y_train = train['Dropped_Course']
X_train = train.drop(columns=['Dropped_Course'])
X_test = test.copy()

בשלב זה הופרד משתנה המטרה (Dropped_Course) מקובץ האימון ונשמר במשתנה נפרד (y_train).
 לאחר מכן נוצר מערך המאפיינים (X_train) המכיל את כלל המשתנים המסבירים בלבד.

הפרדה זו מאפשרת למודל ללמוד את הקשרים בין מאפייני ההזמנה לבין משתנה המטרה מבלי להשתמש בערכי התשובה עצמם במהלך האימון. בנוסף, קובץ המבחן נשמר בנפרד כ-X_test, מאחר שהוא אינו מכיל את משתנה המטרה ונועד להפקת תחזיות בלבד.

In [ ]:
#הסרת מזהים ייחודיים ועמודת התאריך
columns_to_drop = ['Client_ID', 'Agent_ID', 'Company_ID', 'Course_Start_Date']
X_train = X_train.drop(columns=columns_to_drop, errors='ignore')
X_test = X_test.drop(columns=columns_to_drop, errors='ignore')

בשלב זה הסרנו משני הקבצים במקביל את עמודות המזהים הייחודיים (Client_ID, Agent_ID, Company_ID) ואת עמודת התאריך המקורית (Course_Start_Date). מחיקת ה-IDs קריטית כדי למנוע מהמודל לפתח התאמת-יתר (Overfitting) וללמוד "בעל פה" מספרים סידוריים רנדומליים במקום דפוסים אמיתיים. עמודת התאריך הוסרה מאחר שהיא מגיעה כטקסט שהאלגוריתם אינו יודע לקרוא, ולאחר שכבר חילצנו ממנה את המידע שאנחנו צריכים-   Course_Start_Month`.

In [ ]:
#המרת משתנים הקטגוריאליים למשתנים מספריים
X_train_encoded = pd.get_dummies(X_train, drop_first=True)
X_test_encoded = pd.get_dummies(X_test, drop_first=True)

בשלב זה המרנו את כל המשתנים הקטגוריאליים למשתנים מספריים באמצעות הפונקציה pd.get_dummies(). פעולה זו נחוצה משום שמודלי למידת מכונה אינם יכולים לעבוד ישירות עם ערכים טקסטואליים.

ההמרה יוצרת עמודה נפרדת עבור כל קטגוריה אפשרית ומסמנת באמצעות 0 או 1 האם אותה קטגוריה קיימת בתצפית.

בנוסף, השתמשנו בפרמטר drop_first=True, אשר מסיר קטגוריה אחת מכל משתנה. פעולה זו מקטינה את מספר העמודות ומונעת יצירת מידע כפול בנתונים.

In [ ]:
#התאמת מבנה הנתונים בין קובץ האימון לקובץ הבדיקה
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join='left', axis=1, fill_value=0)
print(f"shape - train: {X_train_encoded.shape}")
print(f"shape- test: {X_test_encoded.shape}")

לאחר ביצוע One-Hot Encoding נוצרו עמודות חדשות עבור הקטגוריות השונות בנתונים. מאחר שלא כל הקטגוריות מופיעות גם בקובץ האימון וגם בקובץ הבדיקה, ייתכן שמספר העמודות בשני הקבצים יהיה שונה.

כדי לוודא שלמודל יהיה אותו מבנה נתונים בזמן האימון ובזמן החיזוי, ביצענו התאמה (Alignment) בין הקבצים. פעולה זו דואגת שלשני הקבצים יהיו בדיוק אותן עמודות, ובמקרה שחסרה עמודה בקובץ הבדיקה היא נוצרת אוטומטית ומקבלת את הערך 0.

שלב זה הכרחי כדי למנוע שגיאות ולהבטיח שהמודל יוכל לבצע תחזיות בצורה תקינה.

In [ ]:
from sklearn.model_selection import train_test_split

X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train_encoded, y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42
)
print(f"Train: {X_train_final.shape}, Validation: {X_val.shape}")

בשלב זה חילקנו את נתוני האימון לשתי קבוצות: 80% לאימון המודל (Training Set) ו־20% לסט ולידציה (Validation Set). סט האימון משמש ללמידת המודל, בעוד שסט הוולידציה משמש להערכת ביצועיו על נתונים שלא ראה קודם, וכך מתקבלת הערכה אמינה יותר של יכולת ההכללה של המודל

Part C - Model Training and Hyperparameters Tuning
בחלק זה נתמקט בבנייה, אימון ואופטימיזציה של מודלים שונים ללמידת מכונה. מטרת השלב היא למצוא את המודל בעל כושר החיזוי הגבוה ביותר עבור הסתברות ביטול הקורס, תוך כוונון פרמטרים ומניעת התאמת-יתר (Overfitting)

In [ ]:
from sklearn.model_selection import StratifiedKFold
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)



בקטע קוד זה הוגדרה אסטרטגיית התיקוף באמצעות **Stratified K-Fold Cross-Validation** עם 5 קיפולים (Folds). בשיטה זו, מערך האימון מחולק לחמישה חלקים, כאשר בכל איטרציה המודל מאומן על ארבעה חלקים ונבחן על החלק הנותר, והתהליך חוזר עד שכל חלק משמש כקבוצת בדיקה פעם אחת. השימוש ב-**Stratified** מבטיח שהתפלגות משתנה המטרה **Dropped_Course** תישמר באופן דומה בכל הקיפולים, ובכך מצמצם הטיות הנובעות מחלוקה לא מאוזנת של הנתונים. בנוסף, **shuffle=True** מערבב את התצפיות לפני החלוקה, ו-**random_state=42** מבטיח שהתוצאות יהיו ניתנות לשחזור בכל הרצה מחדש של הקוד.


In [ ]:
#מודל ראשון - Logistic Regression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
log_reg_baseline = LogisticRegression(C=1.0, solver='liblinear', random_state=42, max_iter=1000)
auc_scores = cross_val_score(estimator=log_reg_baseline, X=X_train_final, y=y_train_final, cv=cv_strategy, scoring='roc_auc', n_jobs=-1)
print(f"All Validation AUC Scores: {auc_scores}")
print(f"Mean Validation AUC Score (Baseline): {auc_scores.mean():.4f}")

כמודל ראשוני  בחרנו להשתמש באלגוריתם Logistic Regression, הנחשב למודל נפוץ ומתאים לבעיות סיווג בינארי. המודל מחשב את ההסתברות להשתייכות לכל מחלקה ומהווה נקודת השוואה בסיסית למודלים המתקדמים שנבחנו בהמשך. לצורך האימון נבחר ערך רגולריזציה של C=1.0, המאפשר איזון בין מורכבות המודל לבין יכולת ההכללה שלו, ונעשה שימוש בפותר liblinear המתאים לבעיות סיווג ליניאריות ומספק ביצועים יעילים על מערך הנתונים. הערכת ביצועי המודל בוצעה באמצעות Stratified 5-Fold Cross-Validation, המבטיח שמירה על התפלגות דומה של מחלקות היעד בכל חלוקת תיקוף. המודל הניב ציון AUC ממוצע של 0.8984, כאשר היציבות הגבוהה בין סבבי התיקוף מעידה על יכולת הכללה טובה ועל איכות תהליך הכנת הנתונים והמאפיינים.

In [ ]:
#מודל שני - Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
rf_model = RandomForestClassifier(random_state=42)
param_dist_rf = {
    'n_estimators': [50, 100],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10]
}
random_search_rf = RandomizedSearchCV(
    estimator=rf_model,
    param_distributions=param_dist_rf,
    n_iter=4,
    cv=cv_strategy,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1
)
random_search_rf.fit(X_train_final, y_train_final)

print(f"Best Hyperparameters for RF: {random_search_rf.best_params_}")
print(f"Best Validation AUC Score (RF): {random_search_rf.best_score_:.4f}")

כמודל השני בחרנו להשתמש באלגוריתם Random Forest Classifier, המבוסס על אנסמבל של עצי החלטה ומאפשר זיהוי קשרים מורכבים ולא-ליניאריים בנתונים, אשר עשויים שלא להיתפס על ידי מודלים אחרים. לצורך כוונון המודל ובקרת התאמת-יתר (Overfitting), בוצעה סריקת היפר-פרמטרים באמצעות RandomizedSearchCV, המאפשרת לבחון מגוון רחב של שילובי פרמטרים בצורה יעילה מבחינה חישובית. הערכים שנבחרו היו 50 עצים, עומק מקסימלי של 20 ומינימום של 5 דגימות לפיצול צומת, במטרה לאזן בין יכולת הלמידה של המודל לבין יכולת ההכללה שלו. הערכת ביצועי המודל באמצעות Stratified 5-Fold Cross-Validation הניבה ציון AUC ממוצע של 0.9255. התוצאה מעידה על יכולתו לזהות דפוסים מורכבים במאגר הנתונים.

In [ ]:
#מודל שלישי - XGBClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
xgb_model = XGBClassifier(eval_metric='logloss', random_state=42)
param_dist_xgb = {
    'n_estimators': [50, 100],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.8, 1.0]
    }
random_search_xgb = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_dist_xgb,
    n_iter=4,
    cv=cv_strategy,
    scoring='roc_auc',
    random_state=42,
    n_jobs=-1
)
random_search_xgb.fit(X_train_final, y_train_final)
print(f"Best Hyperparameters for XGBoost: {random_search_xgb.best_params_}")
print(f"Best Validation AUC Score (XGBoost): {random_search_xgb.best_score_:.4f}")

כמודל השלישי בחרנו להשתמש באלגוריתם XGBClassifier, המהווה מימוש של שיטת XGBoost המבוססת על גישת ה-Gradient Boosting. המודל בונה עצי החלטה באופן סדרתי, כך שכל עץ חדש מתמקד בתיקון השגיאות של העצים שנבנו לפניו. לצורך כוונון המודל ומניעת התאמת-יתר (Overfitting), בוצעה סריקת היפר-פרמטרים באמצעות RandomizedSearchCV, במסגרתה נבחנו שילובים שונים של מספר עצים, עומק העצים, קצב הלמידה ושיעור הדגימה. הערכים האופטימליים שנבחרו היו 100 עצים, קצב למידה של 0.2, עומק מקסימלי של 7 ושימוש מלא בדגימות האימון. הערכת ביצועי המודל באמצעות Stratified 5-Fold Cross-Validation הניבה ציון AUC של 0.9445. התוצאה מעידה על יתרונו של המודל בזיהוי דפוסים מורכבים בנתונים.

In [ ]:
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
xgb_base = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
param_grid_fast = {
    'n_estimators': [100, 150],
    'max_depth': [3, 4],
    'learning_rate': [0.05, 0.1]
}

grid_search_fast = GridSearchCV(
    estimator=xgb_base,
    param_grid=param_grid_fast,
    scoring='roc_auc',
    cv=2,
    verbose=1,
    n_jobs=-1
)
grid_search_fast.fit(X_train_final, y_train_final)
best_general_xgb = grid_search_fast.best_estimator_
y_pred_best = best_general_xgb.predict(X_val)
print(f"השילוב היציב שנמצא: {grid_search_fast.best_params_}")
print(f"ציון AUC בתוך ה-Train: {grid_search_fast.best_score_:.4f}")

במסגרת שלב ההערכה (Evaluation) של מתודולוגיית CRISP-DM, אנו בוחנים את המודלים השונים באמצעות מטריצת טעויות (Confusion Matrix) בעזרת 4 מתודות:
*   $TP$ (True Positive): קבוצות שנובאו כמבטלות ואכן ביטלו.
*   $TN$ (True Negative): קבוצות שנובאו כלא מבטלות ואכן לא ביטלו.
*   $FP$ (False Positive): אזעקות שווא – סטודנטים שנובאו כנושרים אך המשיכו בפועל.
*   $FN$ (False Negative): פספוסים – סטודנטים שנובאו כממשיכים אך נשרו בפועל.
על בסיס ארבעת ערכים אלו, הקוד מבוסס על הנוסחאות המתמטיות הבאות לצורך חישוב מדדי האיכות:
*   Accuracy (דיוק כללי): האחוז הכולל של הניבויים הנכונים מתוך כלל המדגם:

 $$Accuracy = \frac{TP+TN}{TP+TN+FP+FN}$$.
* Precision (דיוק החיזוי לחיוביים): מדד הבוחן כמה מתוך אלו שסומנו כנושרים הם אכן נושרים אמיתיים:
$$\text{Precision} = \frac{TP}{TP + FP}$$
*   Recall  (רגישות המודל): מדד הבוחן כמה מתוך סך הסטודנטים שבאמת נשרו במציאות, המודל הצליח לגלות ולאתר מראש:
$$\text{Recall} = \frac{TP}{TP + FN}$$
* F1-Score (מדד משולב): הממוצע ההרמוני המאזן ומקשר בין ה-Precision ל-Recall. הוא מהווה את המדד המרכזי להשוואה כאשר יש חשיבות משותפת לשני סוגי הטעויות:
$$\text{F1-Score} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$












In [ ]:
import pandas as pd
from sklearn.metrics import confusion_matrix
log_reg_baseline.fit(X_train_final, y_train_final)
y_pred_log = log_reg_baseline.predict(X_val)
y_pred_rf = random_search_rf.predict(X_val)
y_pred_best = best_general_xgb.predict(X_val)

def calculate_evaluation_metrics(y_true, y_pred, model_name):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return {
        'Model Name': model_name,
        'Accuracy': round(accuracy, 4),
        'Precision': round(precision, 4),
        'Recall': round(recall, 4),
        'F1-Score': round(f1, 4)
    }

evaluation_results = [
    calculate_evaluation_metrics(y_val, y_pred_log, 'Logistic Regression'),
    calculate_evaluation_metrics(y_val, y_pred_rf, 'Random Forest Classifier'),
    calculate_evaluation_metrics(y_val, y_pred_best, 'XGBoost Classifier')
]

df_evaluation = pd.DataFrame(evaluation_results)
df_evaluation.style.hide(axis='index').highlight_max(
    subset=['Accuracy', 'Precision', 'Recall', 'F1-Score'],
    color='lightgreen'
)

הקוד שלעיל מפיק את הניבויים הבינאריים (0 או 1) עבור שלושת המודלים באמצעות הפקודה .predict() על גבי נתוני האימון המקודדים. לאחר מכן, בעזרת פונקציית confusion_matrix וחילוץ באמצעות .ravel(), הקוד מקבל את ארבעת ערכי היסוד של מטריצת הטעויות ומעביר אותם לפונקציה ייעודית המחשבת ומעגלת את המדדים התיאורטיים (Accuracy, Precision, Recall ו-F1-Score). לבסוף, הנתונים מרוכזים בטבלת DataFrame של ספריית pandas ומעוצבים בצורה ויזואלית הבוחרת וצובעת בירוק בהיר את הערך המקסימלי בכל עמודה כדי להבליט את המודל המוביל.

Logistic Regression

מודל הרגרסיה הלוגיסטית השיג דיוק כללי (Accuracy) של 82.21% וערך F1-Score של 0.7668. המודל הגיע ל-Precision של 0.8393, כלומר כ-84% מהסטודנטים שסווגו כבעלי סיכון לנשירה אכן נשרו בפועל. עם זאת, ערך ה-Recall עמד על 0.7059, דבר המעיד כי המודל מצליח לזהות כ-71% מהקבוצות שביטלו בפועל, אך מפספס כ-29% מהם. תוצאות אלו מצביעות על כך שהמודל מספק ביצועים סבירים, אך הקשרים הליניאריים שעליהם הוא מבוסס אינם מצליחים ללכוד במלואם את המורכבות והאינטראקציות הקיימות בין המשתנים המשפיעים על נשירת סטודנטים.

Random Forest Classifier

מודל ה-Random Forest הציג התנהגות שונה באופן משמעותי. יתרונו הבולט ביותר הוא ערך Precision גבוה של 0.9506 – הגבוה ביותר מבין כל המודלים שנבחנו. משמעות הדבר היא שכאשר המודל מזהה סטודנט כבעל סיכון לביטול, הוא צודק בכ-95% מהמקרים. תוצאה זו מפחיתה משמעותית את מספר ההתערבויות המיותרות מול סטודנטים שאינם צפויים לנשור, ובכך חוסכת משאבים ועלויות. עם זאת, ערך ה-Recall של המודל עומד על 0.6143 בלבד, כלומר המודל מצליח לזהות רק כ-61% מהקבוצות שביטלו בפועל ומפספס כ-39% מהם. מדובר בחיסרון משמעותי, שכן כל סטודנט שנושר מבלי שזוהה מראש מייצג אובדן הכנסה והחמצת הזדמנות לפעולת שימור.

XGBoost Classifier

מודל ה-XGBoost הציג את הביצועים הטובים והמאוזנים ביותר מבין כל המודלים שנבדקו. הוא השיג את רמת הדיוק הגבוהה ביותר (Accuracy = 85.61%), לצד Precision של 0.8795 ו-Recall של 0.7563. המשמעות היא שהמודל מצליח לזהות למעלה מ-75% מהסטודנטים שבאמת צפויים לנשור, תוך שמירה על שיעור נמוך יחסית של אזעקות שווא. שילוב זה מעיד על יכולת גבוהה הן בזיהוי מוקדם של סטודנטים בסיכון והן במניעת התערבויות מיותרות. היתרון המרכזי של XGBoost על פני Random Forest מתבטא באיזון שבין Precision ל-Recall. בעוד ש-Random Forest מתמקד בדיוק גבוה מאוד על חשבון פספוס נושרים רבים, XGBoost מצליח לשמור על רמת דיוק גבוהה תוך זיהוי חלק גדול משמעותית מהקבוצות שביטלו בפועל

סיכום

לצורך בחירת המודל הסופי נבחן גם מדד ה-F1-Score, אשר משלב בין Precision לבין Recall ומספק מדד מאוזן לאיכות הסיווג. מודל ה-XGBoost השיג את הציון הגבוה ביותר עם F1-Score של 0.8132, לעומת 0.7668 עבור Logistic Regression ו-0.7463 עבור Random Forest. לאור תוצאות אלו נבחר מודל ה-XGBoost כמודל הסופי, שכן הוא מספק את האיזון הטוב ביותר בין דיוק בזיהוי סטודנטים בסיכון לבין צמצום פספוסי נשירה, ובכך מתאים ביותר למטרת הפרויקט.

לאחר שלב הערכת המודלים (Model Evaluation) והשוואת הביצועים באמצעות מדדי ההערכה השונים ומטריצת הבלבול (Confusion Matrix), נמצא כי מודל ה-XGBoost Classifier הוא המודל שהציג את הביצועים הטובים והמאוזנים ביותר עבור משימת חיזוי. המודל השיג את ערך ה-F1-Score הגבוה ביותר (0.8132), דבר המעיד על איזון מוצלח בין זיהוי סטודנטים הנמצאים בסיכון לנשירה לבין צמצום תחזיות שגויות.

לאור תוצאות אלו, בחרנו במודל ה-XGBoost כמודל הסופי להמשך הניתוח. בשלב הבא נשתמש בשיטת SHAP, המאפשרת להסביר כיצד המודל מקבל את החלטותיו ולבחון את ההשפעה של כל משתנה על התחזיות שהוא מפיק. באמצעות ניתוח זה נוכל לזהות מהם הגורמים המשמעותיים ביותר המשפיעים על נשירת סטודנטים, להבין טוב יותר את התנהגות המודל, ולהפיק תובנות.



In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
cm_xgb = confusion_matrix(y_val, y_pred_best)
print(cm_xgb)

plt.figure(figsize=(6, 5))
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Did not cancel (0)', 'Cancel (1)'],
            yticklabels=['Did not cancel (0)', 'Cancel (1)'])
plt.title('Confusion Matrix - XGBoost (best model)')
plt.xlabel('Model forecast')
plt.ylabel('Real value')
plt.tight_layout()
plt.savefig('confusion_matrix_xgb.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
from xgboost import plot_importance
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 8))
plot_importance(best_general_xgb, ax=ax, max_num_features=10, importance_type='gain')
plt.title('Feature Importance (Gain) - XGBoost (best_general_xgb)')
plt.xlabel('Average Gain')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
importance_gain = best_general_xgb.get_booster().get_score(importance_type='gain')
top_features = sorted(importance_gain.items(), key=lambda x: x[1], reverse=True)[:10]
for name, score in top_features:
    print(f"{name}: {score:.2f}")

In [ ]:
#התקנת ספריית shape
!pip install shap -q

import shap
import matplotlib.pyplot as plt
best_xgb_model = best_general_xgb
explainer = shap.TreeExplainer(best_xgb_model)
shap_values = explainer(X_train_encoded)
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_train_encoded, max_display=10, show=False)
plt.title('SHAP Summary Plot - XGBoost Feature Importance', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

**ניתוח תוצאות הגרף**

Payment_Terms_prepaid (non-refundable):
 המשתנה המשפיע ביותר על החלטות המודל הוא מסלול התשלום מראש ללא אפשרות להחזר. בגרף ניתן לראות כי ערכים גבוהים של משתנה זה ממוקמים בצורה מובהקת בימין הגרף, ותורמים באופן משמעותי להגדלת הסבירות לנשירה. מבחינה עסקית, ממצא זה מעיד כי לקוחות במסלול קשיח זה נוטים לנשור יותר – תופעה המאפיינת לרוב רכישות אימפולסיביות המונעות מהנחות מחיר, או לחילופין מתסכול וחוסר גמישות של הסטודנט מול המערכת ברגע שנוצרים קשיים לימודיים, דבר המוביל להרמת ידיים ונשירה מוקדמת.

Prev_Course_Dropouts – מספר נשירות מקורסים קודמים
זהו אחד המשתנים החזקים והמשמעותיים ביותר במודל. ניכר כי ככל שלסטודנט קיימת היסטוריה של נשירה מקורסים קודמים, כך עולה באופן משמעותי הסבירות שהמודל יסווג אותו כבעל סיכון גבוה לנשירה נוספת.
הממצא מחזק את ההנחה כי התנהגות עבר מהווה אינדיקטור משמעותי להתנהגות עתידית, ולכן סטודנטים בעלי היסטוריה של נשירה מהווים קבוצת סיכון מרכזית שיש לתת עליה דגש בתהליכי השימור.

Physical_Course_Kits – ערכות קורס פיזיות
המשתנה Physical_Course_Kits נמצא כבעל השפעה משמעותית בכיוון ההפוך. מהגרף עולה כי קבלת ערכת קורס פיזית מפחיתה באופן ניכר את הסיכוי לנשירה.
ניתן להסביר ממצא זה בכך שהערכה הפיזית יוצרת תחושת מחויבות וחיבור לקורס כבר בשלבים הראשונים, ומגבירה את תחושת ההשקעה והמעורבות של הסטודנט בתהליך הלמידה.

Registration_Days_Before – מספר הימים בין ההרשמה לתחילת הקורס
המודל מזהה קשר ברור בין מועד ההרשמה לבין הסיכון לנשירה. סטודנטים שנרשמים זמן רב לפני תחילת הקורס נוטים להציג סיכון גבוה יותר לביטול, בעוד שסטודנטים הנרשמים סמוך למועד תחילת הקורס נוטים להישאר ולהשלים את ההרשמה בפועל.
ייתכן כי פרק זמן ארוך בין ההרשמה לתחילת הלימודים מגדיל את הסיכוי לשינוי תוכניות, אובדן עניין או מעבר לחלופות אחרות.

משתנים כגון התאמת המעבדה (Lab_Config_Match) ועלות הלימוד היומית (Daily_Tuition_Cost) מציגים ערכי SHAP נמוכים יחסית ומרוכזים סביב האפס. משמעות הדבר היא שלמרות שהם תורמים במידה מסוימת לחיזוי, השפעתם על החלטות המודל מוגבלת בהשוואה למשתנים המרכזיים שהוצגו לעיל.
לפיכך, משתנים אלו אינם מהווים בפני עצמם אינדיקטור חזק לנשירה או לשימור, והשפעתם באה לידי ביטוי בעיקר בשילוב עם משתנים נוספים.

**דוגמאות לאופן קבלת ההחלטות של המודל ורמות הביטחון בחיזוי**

באמצעות ערכי ה-SHAP ניתן להבין לא רק אילו משתנים משפיעים על תחזיות המודל, אלא גם כיצד שילובים שונים של מאפיינים משפיעים על רמת הביטחון של החיזוי.

תצפית ברמת ביטחון גבוהה לביטול (High Confidence – Dropout)

נניח שמדובר בסטודנט שנרשם למסלול תשלום מראש ללא החזר, בעל היסטוריה של נשירות מקורסים קודמים, ונרשם זמן רב לפני תחילת הקורס. כל אחד מהמאפיינים הללו תורם באופן חיובי ומשמעותי להגדלת הסיכון לנשירה.

במקרה כזה, מספר גורמי סיכון חזקים פועלים יחד באותו כיוון ומחזקים זה את זה. כתוצאה מכך, המודל יסווג את הסטודנט כבעל סבירות גבוהה מאוד לנשירה ויספק תחזית ברמת ביטחון גבוהה במיוחד.

תצפית ברמת ביטחון גבוהה לשימור (High Confidence – Retention)

לעומת זאת, סטודנט ללא היסטוריה של נשירות, שאינו נמצא במסלול תשלום מראש ללא החזר וקיבל ערכת קורס פיזית, מציג מאפיינים המזוהים על ידי המודל כגורמי שימור משמעותיים.

במקרה זה מרבית המשתנים המרכזיים תורמים להפחתת הסיכון לנשירה, ולכן המודל ינבא שהסטודנט צפוי להישאר בקורס ברמת ביטחון גבוהה.

תצפית ברמת ביטחון נמוכה (Low Confidence)

רמת ביטחון נמוכה מתקבלת כאשר קיימים גורמים מנוגדים המשפיעים על התחזית. לדוגמה, סטודנט בעל היסטוריה של נשירות קודמות, אך במקביל קיבל ערכת קורס פיזית ונרשם סמוך למועד פתיחת הקורס.

במצב כזה חלק מהמשתנים דוחפים את התחזית לכיוון נשירה, בעוד שמשתנים אחרים דוחפים לכיוון שימור. כתוצאה מכך ההשפעות מתקזזות באופן חלקי, והמודל מתקשה להכריע באופן חד-משמעי. במקרים אלו ההסתברות החזויה תהיה קרובה יותר ל-50%, והתחזית תתקבל ברמת ביטחון נמוכה יחסית.

מסקנה

ניתוח זה ממחיש כי מודל ה-XGBoost אינו מסתמך על משתנה בודד, אלא משקלל מספר גורמים במקביל. כאשר רוב הגורמים פועלים באותו כיוון, המודל מספק תחזיות ברמת ביטחון גבוהה. לעומת זאת, כאשר קיימים גורמים סותרים, רמת הביטחון יורדת והמודל מציג חוסר ודאות גדול יותר בהחלטתו.

In [ ]:
y_test_probabilities = best_general_xgb.predict_proba(X_test_encoded)[:, 1]

if 'Client_ID' in test_original.columns:
    real_client_ids = test_original['Client_ID'].values
elif 'client_id' in test_original.columns:
    real_client_ids = test_original['client_id'].values
else:
    real_client_ids = test_original.iloc[:, 0].values

output_df = pd.DataFrame({
    'Client_ID': real_client_ids,
    'Drop_Probability': y_test_probabilities
})

output_df.to_csv('predictions.csv', index=False)
pd.set_option('display.max_rows', None)
display(output_df)

**יצירת קובץ תחזיות והסתברויות לנשירה**

בשלב זה הופק קובץ התחזיות הסופי עבור נתוני הטסט באמצעות מודל ה-XGBoost שנבחר כמודל המוביל בפרויקט. במקום להסתפק בסיווג בינארי של "ינשור" או "לא ינשור", השתמשנו בפונקציה predict_proba, אשר מחשבת עבור כל סטודנט את ההסתברות לנשירה כערך רציף בין 0 ל-1.

לאחר חישוב ההסתברויות, חוברו התוצאות למזהה הסטודנט (client_id) ונוצר קובץ מסודר המכיל את כלל התצפיות ואת ההסתברות החזויה לנשירה עבור כל אחת מהן. לבסוף, הנתונים יוצאו לקובץ Excel לצורך שימוש והמשך ניתוח.

קובץ זה מאפשר ל-Nova Academy לזהות מראש סטודנטים בעלי סיכון גבוה לנשירה, לדרג אותם לפי רמת הסיכון, ולבנות פעולות שימור ממוקדות ויעילות. שימוש בהסתברויות ולא רק בסיווג סופי מספק תמונה מדויקת יותר של רמת הסיכון ומאפשר קבלת החלטות מושכלת המבוססת על נתונים.